In [0]:
dbutils.widgets.text("p_batch_id", "")

In [0]:
v_batch_id = dbutils.widgets.get("p_batch_id")
print(v_batch_id)

In [0]:
%run  ../00-common/01_Environmnet_config

In [0]:
%run ../00-common/02_bronze_helpers

In [0]:
silver_results_table = f"{catalog_name}.{silver_schema}.results"
silver_sprints_table = f"{catalog_name}.{silver_schema}.sprints"
target_table = f"{catalog_name}.{gold_schema}.fact_results"

In [0]:
results_df = (
    spark.read.table(silver_results_table)
    .filter(F.col("batch_id") == v_batch_id)
    .withColumn("session_type", F.lit("RACE"))
    .drop("race_name", "race_date", "ingestion_timestamp", "source_file", "batch_id", "created_timestamp", "updated_timestamp")
)

sprints_df = (
    spark.read.table(silver_sprints_table)
    .filter(F.col("batch_id") == v_batch_id)
    .withColumn("session_type", F.lit("SPRINT"))
    .drop("race_name", "race_date", "ingestion_timestamp", "source_file", "batch_id", "created_timestamp", "updated_timestamp")
)

In [0]:
dim_df = results_df.unionByName(sprints_df)

In [0]:
%sql
DROP TABLE IF EXISTS formula1_incr.gold.fact_results;

In [0]:
fact_session_results_df = dim_df.withColumns(
    {
        "is_win": F.when(F.col("finish_position") == 1, True).otherwise(False),
        "is_podium": F.when(F.col("finish_position").between(1, 3), True).otherwise(
            False
        ),
        "has_points": F.when(F.col("points") > 0, True).otherwise(False),
    }
)

In [0]:
write_to_gold(
    fact_session_results_df,
    target_table,
    """ 
    t.season = s.season
    AND t.round = s.round
    AND t.driver_id = s.driver_id
    AND t.session_type = s.session_type
    AND t.constructor_id = s.constructor_id
    """,
    columns_to_update=[
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "finish_position",
        "finish_position_text",
        "status",
        "is_win",
        "is_podium",
        "has_points",
    ],
)

In [0]:
%sql
select
  *
from
  formula1_incr.gold.fact_results